In [36]:
# import functions from unified_get_segment.py below
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
from shapely.ops import linemerge
import pandas as pd
from unified_get_segment import compute_point_stationing

In [38]:
nb_df = pd.read_excel('data/I35W_detectors/NTE 35W 1min GP.xlsx', sheet_name='NB_detectors')
# convert nb_df to a geopandas geodataframe using the columns 'longitude' and 'latitude' as the geometry
nb_gdf = gpd.GeoDataFrame(nb_df, geometry=gpd.points_from_xy(nb_df.long, nb_df.lat))
nb_gdf = nb_gdf.set_crs("EPSG:4326")
nb_gdf.head()

,northbound detectors with data,lat,long,geometry
0,MVD-5217E-01,32.749166,-97.319329,POINT (-97.31933 32.74917)
1,MVD-5220E-02,32.762460,-97.317584,POINT (-97.31758 32.76246)
2,MVD-5222E-02,32.770929,-97.318301,POINT (-97.3183 32.77093)
3,MVD-5226E-02,32.778642,-97.319657,POINT (-97.31966 32.77864)
4,MVD-5229E-02,32.784243,-97.320291,POINT (-97.32029 32.78424)


In [28]:
sb_df = pd.read_excel('data/I35W_detectors/NTE 35W 1min GP.xlsx', sheet_name='SB_detectors')
# convert sb_df to a geopandas geodataframe using the columns 'longitude' and 'latitude' as the geometry
sb_gdf = gpd.GeoDataFrame(sb_df, geometry=gpd.points_from_xy(sb_df.long, sb_df.lat))
sb_gdf = sb_gdf.set_crs("EPSG:4326")
sb_gdf.head()

,southbound detectors with data,lat,long,geometry
0,MVD-5217E-02,32.749166,-97.319329,POINT (-97.31933 32.74917)
1,MVD-5219W-03,32.762674,-97.318663,POINT (-97.31866 32.76267)
2,MVD-5223W-02,32.771023,-97.319292,POINT (-97.31929 32.77102)
3,MVD-5226E-05,32.778408,-97.320421,POINT (-97.32042 32.77841)
4,MVD-5229E-05,32.784196,-97.321086,POINT (-97.32109 32.7842)


In [39]:
import geopandas as gpd
# load in the csv at outputs/intermediates/trimmed_segments.csv and convert it to a geodataframe using the column 'geometry' as the geometry
nb_mainline_gdf = pd.read_csv('outputs/intermediates/trimmed_segments.csv')
nb_mainline_gdf['geometry'] = nb_mainline_gdf['geometry'].apply(wkt.loads)
nb_mainline_gdf = gpd.GeoDataFrame(nb_mainline_gdf, geometry='geometry')
nb_mainline_gdf = nb_mainline_gdf.set_crs("EPSG:4326")
nb_mainline_gdf.head()

,Unnamed: 0,way_id,name,ref,highway,lanes,maxspeed,geometry,hov
0,8,1110279956,North South Freeway,I 35W;US 287 Bus,motorway,4,50 mph,"LINESTRING (-97.3195 32.74914, -97.31957 32.75...",NaN
1,9,1335556200,North South Freeway,I 35W;US 287 Bus,motorway,3,50 mph,"LINESTRING (-97.31957 32.75018, -97.31962 32.7...",NaN
2,10,1335556201,North South Freeway,I 35W;US 287 Bus,motorway,3,50 mph,"LINESTRING (-97.31962 32.75079, -97.31963 32.7...",NaN
3,11,122732977,North South Freeway,I 35W;US 287 Bus,motorway,3,50 mph,"LINESTRING (-97.31963 32.75133, -97.31965 32.7...",NaN
4,12,565787784,North South Freeway,I 35W;US 377;US 287 Bus,motorway,4,50 mph,"LINESTRING (-97.31958 32.75371, -97.31953 32.7...",NaN


In [40]:
# assign a variable called start_point with the coordinates of the start of the first row's linestring in nb_mainline_gdf
start_point = nb_mainline_gdf.geometry.iloc[0].coords[0]

# assign a variable called mainline_length with the length in miles of the combined linestrings in nb_mainline_gdf
combined = nb_mainline_gdf.geometry.union_all()
merged_nb_line = linemerge(combined)
merged_proj_nb = gpd.GeoSeries([merged_nb_line], crs="EPSG:4326").to_crs("EPSG:2277").iloc[0]
mainline_length = merged_proj_nb.length / 5280  # convert feet to miles
print(mainline_length)
stationed_nb_detectors = compute_point_stationing(nb_gdf, nb_mainline_gdf, start_point, 0, "EPSG:2277", mainline_length)
stationed_nb_detectors.head(40)

17.60215158459047


,northbound detectors with data,lat,long,geometry,x_rcs,x_rcs_miles
0,MVD-5217E-01,32.749166,-97.319329,POINT (-97.31933 32.74917),6.791352,0.001286
31,MVD-7044W-01,32.750944,-97.313935,POINT (-97.31394 32.75094),554.376074,0.104995
1,MVD-5220E-02,32.762460,-97.317584,POINT (-97.31758 32.76246),4909.806550,0.929888
2,MVD-5222E-02,32.770929,-97.318301,POINT (-97.3183 32.77093),8032.536955,1.521314
3,MVD-5226E-02,32.778642,-97.319657,POINT (-97.31966 32.77864),10872.480658,2.059182
4,MVD-5229E-02,32.784243,-97.320291,POINT (-97.32029 32.78424),12924.356255,2.447795
5,MVD-5236E-01,32.801154,-97.319944,POINT (-97.31994 32.80115),19184.782217,3.633481
6,MVD-5238E-01,32.807202,-97.315733,POINT (-97.31573 32.8072),21742.979303,4.117989
7,MVD-5240E-01,32.813277,-97.313841,POINT (-97.31384 32.81328),24054.193306,4.555718
8,MVD-5242W-02,32.819884,-97.312843,POINT (-97.31284 32.81988),26489.390695,5.016930


In [41]:
stationed_nb_detectors.to_csv('data/I35W_detectors/stationed_nb_detectors.csv', index=False)